# 00 --- Tier 0: verification**Run this before anything else, and before spending a single compute unit.**It catches the one failure mode that costs the most: a label, affine, channel-orderor intensity inconsistency that trains without error and silently invalidatesevery result.Nothing here needs a GPU.

## 0.1 --- Environment and repo

In [ ]:
# 0.1 --- environment!pip -q install "monai==1.4.0" einops nibabel scipy scikit-image!nvidia-smi --query-gpu=name,memory.total --format=csvfrom google.colab import drive; drive.mount('/content/drive')import os, sys, subprocessREPO = '/content/499A'if not os.path.exists(REPO):    !git clone https://github.com/ahnaf-csg/499A---3d-Tumor-Segmentation.git {REPO}else:    subprocess.run(['git','-C',REPO,'pull'])sys.path.insert(0, REPO)import torch, monaiprint('torch', torch.__version__, '| monai', monai.__version__, '| cuda', torch.cuda.is_available())assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'cap = torch.cuda.get_device_capability(0)print(f'SM {cap[0]}.{cap[1]}  bf16={"yes" if cap[0]>=8 else "NO -> fp16 path (T4)"}')

## 0.2 --- Stage the dataTraining off mounted Drive is badly slow for many small reads: BraTS 2021 is~6,250 files of ~2.5 MB. The fix is a one-time tar, after which each sessioncopies ONE large sequential file and untars locally in minutes.Run 0.2a **once**. Every later session runs 0.2b only.

In [ ]:
# 0.2a --- ONE TIME: archive Drive -> single tar (skip if TARs already exist)DRIVE = '/content/drive/MyDrive/Colab Notebooks/499a'import osfor name in ['BraTS2021_Training_Data', 'MU-Glioma-Post']:    tar = f'{DRIVE}/{name}.tar'    if os.path.exists(tar):        print('exists, skipping:', tar); continue    if not os.path.exists(f'{DRIVE}/{name}'):        print('source missing, skipping:', name); continue    print('archiving', name, '(this takes a while, once)')    !tar -cf "{tar}" -C "{DRIVE}" "{name}"    !ls -lh "{tar}"

In [ ]:
# 0.2b --- EVERY SESSION: copy tar -> local disk, untarDRIVE = '/content/drive/MyDrive/Colab Notebooks/499a'LOCAL = '/content/data'!mkdir -p {LOCAL}import os, timefor name in ['BraTS2021_Training_Data']:      # add 'MU-Glioma-Post' for Tier 7    if os.path.exists(f'{LOCAL}/{name}'):        print('already local:', name); continue    t0 = time.time()    !cp "{DRIVE}/{name}.tar" /content/    !tar -xf /content/{name}.tar -C {LOCAL}    !rm /content/{name}.tar    print(f'{name}: staged in {time.time()-t0:.0f}s')!du -sh {LOCAL}/*

## 0.3 --- Run the verifierExpect BraTS labels `{0,1,2,4}` (ET=4) and MU `{0,1,2,3,4}` (ET=3, RC=4).**If either differs, stop and fix `datasets.py` before training** — everyregion would otherwise be silently wrong.

In [ ]:
from glioseg.verify import run_alllog, out = run_all('/content/data',                   datasets=('brats2021',),      # add 'mu_post' once staged                   sample=20,                   log_path='/content/drive/MyDrive/Colab Notebooks/499a/results/verification_log.jsonl')

## 0.4 --- Freeze the splitSubject-level, seed 42, written to Drive. Every later run reads this same file, so all arms are compared on identical data.

In [ ]:
from glioseg.config import Configfrom glioseg.datasets import REGISTRY, find_casesfrom glioseg.data import make_split, save_splitfrom glioseg.verify import check_split, LogDRIVE = '/content/drive/MyDrive/Colab Notebooks/499a'SPLIT = f'{DRIVE}/results/split_brats2021.json'cfg = Config(dataset='brats2021', data_base='/content/data', seed=42)cases = find_cases(REGISTRY['brats2021'], cfg.data_base, verbose=False)split = make_split(cases, cfg)save_split(split, SPLIT)check_split(cases, split, Log('/content/drive/MyDrive/Colab Notebooks/499a/results/verification_log.jsonl'))print('\nsplit saved ->', SPLIT)

## 0.5 --- Arm sanity checkShape round-trip, parameter count and peak VRAM for every arm, before committing GPU hours. MedNeXt will report an ImportError until `scripts/vendor_mednext.sh` is run — that is expected and it is the optional 5th arm.

In [ ]:
import pandas as pdfrom glioseg.config import Configfrom glioseg.models import summarize_arm, shape_checkrows = []for m in ['segresnet','unet3d','segformer3d','swinunetr','mednext']:    c = Config(model=m, patch_size=(64,64,64), batch_size=2)    try:        rows.append({**shape_check(c), **summarize_arm(c)})    except Exception as e:        rows.append({'model': m, 'ok': False, 'error': f'{type(e).__name__}: {e}'})pd.DataFrame(rows)

### GateProceed only when: labels match expectations, nesting holds, no split overlap,and at least 4 arms show `ok=True`. Anything red gets fixed here, not later.